In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

# Load model and tokenizer
model_name = "yashpwr/resume-ner-bert-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)

In [ ]:
# Example resume text
text = "John Smith is a senior software engineer with 8 years of experience at Google. He has expertise in Python, JavaScript, and machine learning. Contact: john.smith@gmail.com"

# Tokenize
inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=128,
    padding=True
)
print(inputs)

In [ ]:
# Predict
with torch.no_grad():
    outputs = model(**inputs)
    predictions = torch.argmax(outputs.logits, dim=2)
    print(predictions)

In [ ]:
# Extract entities
entities = []
current_entity = None

for i, pred in enumerate(predictions[0]):
    label = model.config.id2label[pred.item()]
    token = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0][i].item())

    if label.startswith('B-'):
        if current_entity:
            entities.append(current_entity)
        current_entity = {
            'text': token,
            'label': label[2:],  # Remove 'B-' prefix
            'start': i
        }
    elif label.startswith('I-') and current_entity:
        current_entity['text'] += ' ' + token
    elif label == 'O':
        if current_entity:
            entities.append(current_entity)
            current_entity = None

In [ ]:
if current_entity:
    entities.append(current_entity)

print("Extracted Entities:")
for entity in entities:
    print(f"- {entity['label']}: {entity['text']}")


In [ ]:
from transformers import pipeline

# Create NER pipeline
ner_pipeline = pipeline(
    "token-classification",
    model="yashpwr/resume-ner-bert-v2",
    aggregation_strategy="simple"
)

# Extract entities
text = "Sarah Johnson holds a Master's degree in Computer Science from Stanford University. Skills: Python, TensorFlow, SQL."
results = ner_pipeline(text)

for entity in results:
    print(f"{entity['entity_group']}: {entity['word']} (confidence: {entity['score']:.3f})")


In [ ]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "nimendraai/NuExtract-tiny-Resume-Data-Extractor"
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.bfloat16, trust_remote_code=True
).eval().cuda()
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

TEMPLATE = json.dumps({
    "name": "", "email": "", "phone": "", "website": "",
    "skills": [""],
    "experience": [{"title": "", "company": "", "duration": ""}],
    "education":  [{"degree": "", "institution": "", "year": ""}],
    "other_details": [""],
}, indent=4)

def extract_first_json(text):
    depth, start = 0, None
    for i, ch in enumerate(text):
        if ch == "{":
            if start is None: start = i
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0 and start is not None:
                return text[start:i+1]
    return text

def extract(resume_text: str) -> dict:
    prompt = (
        "<|input|>\n"
        f"### Template:\n{TEMPLATE}\n"
        f"### Text:\n{resume_text}\n\n"
        "<|output|>"
    )
    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=2048
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=512, do_sample=False
        )
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    raw = decoded.split("<|output|>")[-1].strip()
    return json.loads(extract_first_json(raw))


In [ ]:
import re
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

MODEL_NAME = "yashpwr/resume-ner-bert-v2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME)

ner = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)

EMAIL_RE = re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b")
PHONE_RE = re.compile(r"(?:(?:\+?\d{1,3}[\s-]?)?(?:\(?\d{2,5}\)?[\s-]?)?\d{3,5}[\s-]?\d{4,})")
URL_RE = re.compile(r"\b(?:https?://|www\.)\S+\b", re.I)

DEGREE_KEYWORDS = {
    "b.tech", "btech", "b.e", "be", "b.sc", "bsc", "bca",
    "m.tech", "mtech", "m.e", "me", "m.sc", "msc", "mca",
    "bachelor", "master", "phd", "doctorate", "mba"
}

SKILL_KEYWORDS = {
    "python", "java", "javascript", "typescript", "sql", "mysql", "postgresql",
    "spring", "spring boot", "react", "node", "tensorflow", "pytorch",
    "docker", "kubernetes", "aws", "gcp", "azure", "git", "linux",
    "html", "css", "tailwind", "mongodb", "pandas", "numpy"
}

SECTION_HINTS = {
    "skills": "Skills",
    "technical skills": "Skills",
    "education": "Education",
    "experience": "Experience",
    "work experience": "Experience",
    "projects": "Projects",
    "certifications": "Certifications"
}


def normalize_text(s: str) -> str:
    s = s.strip()
    s = re.sub(r"\s+([,.;:])", r"\1", s)
    s = re.sub(r"([(\[]) ", r"\1", s)
    s = re.sub(r" ([)\]])", r"\1", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip(" ,;:-")


def is_probable_email(text: str) -> bool:
    return bool(EMAIL_RE.fullmatch(text.strip()))


def is_probable_phone(text: str) -> bool:
    digits = re.sub(r"\D", "", text)
    return len(digits) >= 10 and bool(PHONE_RE.search(text))


def is_probable_degree(text: str) -> bool:
    t = text.lower()
    return any(k in t for k in DEGREE_KEYWORDS)


def split_skills(span_text: str):
    text = normalize_text(span_text)
    parts = re.split(r"[,\n|/•]+", text)
    cleaned = []
    for p in parts:
        p = normalize_text(p).lower()
        if not p:
            continue
        if p in SECTION_HINTS:
            continue
        if p.startswith("skills"):
            p = p.replace("skills", "", 1).strip(" :")
        if p and (p in SKILL_KEYWORDS or len(p.split()) <= 3):
            cleaned.append(p.title() if p not in {"sql", "aws", "gcp", "css", "html"} else p.upper())
    return list(dict.fromkeys(cleaned))


def is_bad_span(text: str, label: str) -> bool:
    t = normalize_text(text)

    if len(t) < 2:
        return True

    if label == "Degree":
        if len(t.split()) > 12:
            return True
        if not is_probable_degree(t):
            return True

    if label == "Skills":
        if len(t.split()) > 15:
            return True

    if label in {"Name", "Company", "College"}:
        if len(t.split()) > 8:
            return True

    if "@" in t and label not in {"Email"}:
        return True

    return False


def extract_rule_based_fields(text: str):
    data = {
        "emails": list(dict.fromkeys(EMAIL_RE.findall(text))),
        "phones": [],
        "links": list(dict.fromkeys(URL_RE.findall(text)))
    }

    phones = []
    for match in PHONE_RE.finditer(text):
        raw = normalize_text(match.group())
        digits = re.sub(r"\D", "", raw)
        if len(digits) >= 10:
            phones.append(raw)

    data["phones"] = list(dict.fromkeys(phones))
    return data


def extract_resume_entities(text: str, threshold_map=None):
    if threshold_map is None:
        threshold_map = {
            "Name": 0.80,
            "Email": 0.70,
            "Phone": 0.70,
            "Skills": 0.75,
            "Degree": 0.80,
            "College": 0.75,
            "Company": 0.75,
            "Designation": 0.75,
        }

    raw_entities = ner(text)
    extracted = {
        "Name": [],
        "Email": [],
        "Phone": [],
        "Skills": [],
        "Degree": [],
        "College": [],
        "Company": [],
        "Designation": [],
        "Other": []
    }

    for ent in raw_entities:
        label = ent["entity_group"]
        score = float(ent["score"])
        span = normalize_text(ent["word"])

        if label not in extracted:
            label = "Other"

        min_score = threshold_map.get(label, 0.75)
        if score < min_score:
            continue

        if is_bad_span(span, label):
            continue

        if label == "Email":
            if is_probable_email(span):
                extracted["Email"].append(span)
            continue

        if label == "Phone":
            if is_probable_phone(span):
                extracted["Phone"].append(span)
            continue

        if label == "Degree":
            if is_probable_degree(span):
                extracted["Degree"].append(span)
            continue

        if label == "Skills":
            extracted["Skills"].extend(split_skills(span))
            continue

        extracted[label].append(span)

    rb = extract_rule_based_fields(text)

    for email in rb["emails"]:
        if email not in extracted["Email"]:
            extracted["Email"].append(email)

    for phone in rb["phones"]:
        if phone not in extracted["Phone"]:
            extracted["Phone"].append(phone)

    extracted["Skills"] = list(dict.fromkeys(extracted["Skills"]))
    for key in extracted:
        extracted[key] = list(dict.fromkeys([normalize_text(x) for x in extracted[key] if normalize_text(x)]))

    return extracted


if __name__ == "__main__":
    text = """
    Sarah Johnson
    Email: sarah.johnson@gmail.com
    Phone: +1 555-123-4567
    LinkedIn: https://linkedin.com/in/sarahjohnson

    Summary:
    Marketing manager with 10 years of experience at Coca-Cola.

    Education:
    Master's degree in Computer Science, Stanford University

    Skills:
    Python, TensorFlow, SQL, React, AWS
    """

    result = extract_resume_entities(text)
    for k, v in result.items():
        if v:
            print(f"{k}: {v}")